In [ ]:
import json
import numpy as np
from sklearn.decomposition import PCA
import hdbscan
import os
from collections import Counter
from tqdm import tqdm
import time
import umap

with open("../embeddings/chunk_embeddings_tedex.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} chunk embeddings")

print("Converting embeddings to np array")
embeddings = np.array([d["embedding"] for d in tqdm(data)])

print("Reducing dimensionality with PCA (retain 95% variance):")
pca = PCA(n_components=0.95, svd_solver="full", random_state=42)
start = time.time()
pca_embeddings = pca.fit_transform(embeddings)
end = time.time()
explained_variance = np.sum(pca.explained_variance_ratio_)
print(f"Total variance retained by PCA: {explained_variance:.4f}")
print(f"PCA reduced dimensions from {embeddings.shape[1]} to {pca_embeddings.shape[1]} in {end - start:.2f} seconds")


print("Reducing dimensionality with UMAP to 20 dimensions:")
reducer = umap.UMAP(n_components=20, metric='euclidean', random_state=42, verbose=True)
start = time.time()
umap_embeddings = reducer.fit_transform(pca_embeddings)
end = time.time()
print(f"UMAP reduced dimensions from {pca_embeddings.shape[1]} to {umap_embeddings.shape[1]} in {end - start:.2f} seconds")

print("Clustering with hdbscan:")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5, 
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
start = time.time()
cluster_labels = clusterer.fit_predict(umap_embeddings)
end = time.time()
print(f"HDBSCAN clustering done in {end - start:.2f} seconds")

for d, label in zip(data, cluster_labels):
    d["cluster"] = int(label)

print(Counter(cluster_labels))

output_path = "../clusters/chunk_clusters_tedex.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print(f"Saved clustered chunks to {output_path}")
num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_points = np.sum(cluster_labels == -1)
print(f"Identified {num_clusters} clusters with {noise_points} noise points")

C:\Users\ved\Desktop\completion-meter\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 91940 chunk embeddings
Converting embeddings to np array


100%|████████████████████████████████████████████████████████████████████████████████████| 91940/91940 [00:00<00:00, 1534084.32it/s]

Reducing dimensionality with PCA (retain 95% variance):


Total variance retained by PCA: 0.9504
PCA reduced dimensions from 384 to 240 in 7.84 seconds
Reducing dimensionality with UMAP to 20 dimensions:


...

	completed  180  /  200 epochs


Epochs completed: 100%| ████████████████████████████████████████████████████████████████████████████████████████████ 200/200 [04:10]


Fri May 30 15:00:38 2025 Finished embedding
UMAP reduced dimensions from 240 to 20 in 374.53 seconds
Clustering with hdbscan:


C:\Users\ved\Desktop\completion-meter\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\ved\Desktop\completion-meter\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


HDBSCAN clustering done in 40.03 seconds
Counter({np.int64(-1): 54421, np.int64(1805): 925, np.int64(1240): 656, np.int64(955): 477, np.int64(1723): 405, np.int64(668): 374, np.int64(1770): 288, np.int64(522): 255, np.int64(1664): 221, np.int64(926): 217, np.int64(1708): 217, np.int64(716): 216, np.int64(1406): 212, np.int64(1566): 204, np.int64(621): 202, np.int64(1613): 197, np.int64(911): 188, np.int64(28): 185, np.int64(1281): 172, np.int64(1001): 169, np.int64(1427): 163, np.int64(1161): 162, np.int64(1207): 162, np.int64(718): 154, np.int64(815): 151, np.int64(1096): 148, np.int64(1745): 142, np.int64(1819): 140, np.int64(450): 139, np.int64(1373): 138, np.int64(1441): 132, np.int64(22): 130, np.int64(1679): 126, np.int64(905): 121, np.int64(1599): 120, np.int64(1528): 120, np.int64(1295): 119, np.int64(799): 119, np.int64(444): 119, np.int64(707): 117, np.int64(1674): 117, np.int64(1315): 114, np.int64(773): 114, np.int64(1327): 113, np.int64(1532): 113, np.int64(1059): 113, np.